# Trip planner

**Planner Agent** — Brave, Fetch, weather MCP → markdown report.  
**Reviewer Agent** — no tools; `passed` / `reason` only. If not passed, we stop (no email).  
**Notifier Agent** — email + push MCP → send push notification, process email → HTML + subject + send.

In [ ]:
!uv pip install -q mailtrap

### Imports

In [ ]:
import os
from collections.abc import AsyncIterator
from contextlib import AsyncExitStack
from datetime import datetime
import gradio as gr
from dotenv import load_dotenv
from openai.types.responses.response_text_delta_event import ResponseTextDeltaEvent

from pydantic import BaseModel, Field

from agents import Agent, Runner, trace, ModelSettings
from agents.exceptions import MaxTurnsExceeded
from agents.mcp import MCPServerStdio
from agents.result import RunResultStreaming
from agents.stream_events import RawResponsesStreamEvent, RunItemStreamEvent

### Key loading and MCP params setting

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set")
brave_api_key = os.getenv("BRAVE_API_KEY")
if not brave_api_key:
    raise ValueError("BRAVE_API_KEY is required for Brave Search MCP")

mcp_planner = [
        {"command": "uv", "args": ["run", "weather_server.py"]},
        {"command": "uvx", "args": ["mcp-server-fetch"]},
        {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-brave-search"],
            "env": {"BRAVE_API_KEY": brave_api_key},
        },
]

mcp_notifier = [
    {"command": "uv", "args": ["run", "email_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
]


AGENT_MODEL = "gpt-4o-mini"
MAX_AGENT_TURNS = 30


### Agents Declarations

In [ ]:
class ReviewVerdict(BaseModel):
    passed: bool = Field(description="True if the report is good enough to send to the user")
    reason: str = Field(description="One or two sentences: why pass or what is wrong")


def planner_agent(mcp_servers) -> Agent:
    instructions = f"""You are a trip researcher. Now: {datetime.now():%Y-%m-%d %H:%M}.

Use Brave search, fetch, and the weather tool. If no destination is given, suggest 2–3 options and develop one.
Cover: overview, weather, activities, sites, what not to do, best times to visit, myths, culture, history, food, drinks, nightlife, shopping, transport, costs, safety when applicable.
Output one detailedmarkdown report with sections and a short "Sources consulted" list.
No email or push."""

    return Agent(
        name="TripPlanner",
        instructions=instructions,
        model=AGENT_MODEL,
        mcp_servers=mcp_servers,
        model_settings=ModelSettings(
            temperature=0.7,
            max_tokens=4000,
        ),
    )


def reviewer_agent() -> Agent:
    return Agent(
        name="ReportReviewer",
        instructions="You only judge the trip report against the user's request. Pass if it is complete, sensible, and safe. Fail if thin, wrong topic, or risky advice.",
        output_type=ReviewVerdict,
        model=AGENT_MODEL,
        model_settings=ModelSettings(
            temperature=0.5,
            max_tokens=800,
        ),
    )


def notifier_agent(mcp_servers) -> Agent:
    return Agent(
        name="Notifier",
        instructions="Turn the approved markdown into a clean HTML email (inline CSS), subject under 78 characters. Call push once, then send_email once. End with a one-line confirmation.",
        model=AGENT_MODEL,
        mcp_servers=mcp_servers,
        model_settings=ModelSettings(
            temperature=0.5,
            max_tokens=4000,
        ),
    )


def _tool_name_from_call(item) -> str:
    raw = getattr(item, "raw_item", None)
    if raw is None:
        return "tool"
    name = getattr(raw, "name", None)
    return str(name) if name else "tool"


async def _stream_run(streamed: RunResultStreaming) -> AsyncIterator[str]:
    try:
        async for ev in streamed.stream_events():
            if isinstance(ev, RawResponsesStreamEvent) and isinstance(ev.data, ResponseTextDeltaEvent):
                yield ev.data.delta
            elif isinstance(ev, RunItemStreamEvent) and ev.item.type == "tool_call_item":
                yield f"\n\n*{_tool_name_from_call(ev.item)}*\n\n"
    except MaxTurnsExceeded:
        yield "\n\n*(Stopped: max turns)*\n"

### Input Guardrail

In [ ]:
def validate_input(query: str) -> None:
    user_input = query.strip()
    if not user_input:
        raise ValueError("Describe the trip you want.")
    if len(user_input) > 500:
        raise ValueError("Message too long (max 500 characters).")
    if len(user_input) < 10:
        raise ValueError("Please provide more detail about your trip.")
    low = user_input.lower()
    injection_phrases = (
        "ignore previous",
        "ignore all instructions",
        "system prompt",
        "developer mode",
        "jailbreak",
        "dan mode",
    )
    
    if any(x in low for x in injection_phrases):
        raise ValueError("That request cannot be processed.")

### Trip Ochestration

In [ ]:
async def plan_trip(user_message: str) -> AsyncIterator[str]:
    try:
        user_message = user_message.strip()
        validate_input(user_message)
    except ValueError as e:
        yield str(e)
        return

    with trace("Trip Planner"):
        research_text = ""

        async with AsyncExitStack() as stack:
            servers = [
                await stack.enter_async_context(MCPServerStdio(p, client_session_timeout_seconds=120))
                for p in mcp_planner
            ]
            agent = planner_agent(servers)
            streamed = Runner.run_streamed(agent, user_message, max_turns=20)
            yield "### Research\n\n"
            async for chunk in _stream_run(streamed):
                yield chunk
            research_text = streamed.final_output
            if research_text is None:
                research_text = ""
            elif not isinstance(research_text, str):
                research_text = str(research_text)

        review_prompt = f"User request:\n{user_message}\n\nReport:\n{research_text}"
        rr = await Runner.run(reviewer_agent(), review_prompt, max_turns=10)
        verdict = rr.final_output
        if not isinstance(verdict, ReviewVerdict):
            verdict = ReviewVerdict(passed=False, reason="Review did not return a valid verdict.")
        status = "Passed" if verdict.passed else "Not approved"
        yield f"\n\n---\n\n### Review\n\n**{status}:** {verdict.reason}\n\n"
        if not verdict.passed:
            return

        notify_prompt = f"User request:\n{user_message}\n\nApproved report:\n{research_text}"
        async with AsyncExitStack() as stack:
            servers = [
                await stack.enter_async_context(MCPServerStdio(p, client_session_timeout_seconds=120))
                for p in mcp_notifier
            ]
            agent = notifier_agent(servers)
            streamed = Runner.run_streamed(agent, notify_prompt, max_turns=10)
            yield "### Notify\n\n"
            async for chunk in _stream_run(streamed):
                yield chunk

### Gradio UI intialization

In [ ]:
async def chat_fn(message: str, history: list):
    history = list(history or [])
    history.append([message, ""])
    yield history
    try:
        async for chunk in plan_trip(message):
            history[-1][1] += chunk
            yield history
    except Exception as e:
        history[-1][1] += f"\n\n**Error:** {e!s}"
        yield history

with gr.Blocks(title="Trip planner", theme=gr.themes.Soft(primary_hue="teal")) as ui:
    gr.Markdown(
        "# Trip planner\n"
        "Tell us about your trip and we would plan it for you."
    )
    chatbot = gr.Chatbot(label="Chat", height=480)
    msg = gr.Textbox(
        label="Your trip",
        placeholder="e.g. Long weekend in Lisbon — food and walking",
        lines=2,
    )
    with gr.Row():
        send = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear")

    msg.submit(chat_fn, [msg, chatbot], [chatbot]).then(lambda: gr.update(value=""), outputs=msg)
    send.click(chat_fn, [msg, chatbot], [chatbot]).then(lambda: gr.update(value=""), outputs=msg)
    clear.click(lambda: ([], gr.update(value="")), None, [chatbot, msg])


ui.launch(inbrowser=True)